# Week 5 - Q3: Where is cancellation concentrated?
**Question.** 9.8% of orders are cancelled. Is cancellation concentrated by region, customer type, sales channel or product?

**Approach.** The unit of analysis is the **order** (20,000 orders, one row each), because every line of an order shares the same status. A line-level test would count each order about 6.5 times and overstate the evidence (the same mistake corrected for purchase orders in Q8). Each cut is tested with a chi-square test, reported with an effect size (Cramér's V) and a confidence interval for every segment, and all tests are adjusted together (Holm) so that looking at many cuts does not manufacture a finding.

**Definitions (same as Weeks 3 and 4).** Cancelled order = `order_status == "Cancelled"`. Order value = `total_order_value` (ex-GST, the sum of the order's lines). Region = the customer's region unless stated. Built on the corrected Week 2 merged datasets. **Limit:** the data has no cancellation-reason field, so this notebook finds *where* cancellation sits, not *why* it happens.

In [1]:
import pandas as pd, numpy as np, os
from pathlib import Path
from scipy import stats
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
SURF,INK,INK2,GRID,BLUE,ORANGE,AQUA,MUTED="#fcfcfb","#0b0b0b","#52514e","#e4e3df","#2a78d6","#eb6834","#1baf7a","#b9b8b2"
def find_dir(env_var, marker, rels):
    """Environment variable first, then this folder and every parent folder, using the repository layout."""
    if os.environ.get(env_var) and (Path(os.environ[env_var]) / marker).exists():
        return Path(os.environ[env_var])
    for base in [Path.cwd(), *Path.cwd().parents]:
        for rel in rels:
            if (base / rel / marker).exists():
                return base / rel
    return None
DATA_DIR = find_dir("CADETX_MERGED", "sales_orders_full.csv", ["merged", "data/merged", "week-02/reports/week_02/merged", "reports/week_02/merged"])
assert DATA_DIR is not None, "Could not find the merged datasets. Run from inside the repository, put a merged/ folder next to the notebook, or set the CADETX_MERGED environment variable to the folder."
OUT=Path("q3_outputs"); OUT.mkdir(exist_ok=True); CH=Path("q3_charts"); CH.mkdir(exist_ok=True)
SEED=42; NBOOT=5000
checks=[]
def check(name,expected,actual,ok):
    checks.append(dict(Check_Item=name,Expected=expected,Actual=actual,Status="PASS" if ok else "FAIL")); assert ok,f"{name}: expected {expected}, got {actual}"
def wilson(x,n,z=1.959964):
    p=x/n; d=1+z*z/n; c=(p+z*z/(2*n))/d; h=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/d; return c-h,c+h
def holm(p):
    p=np.asarray(p,float); m=len(p); o=np.argsort(p); adj=np.empty(m); run=0
    for r,i in enumerate(o): run=max(run,(m-r)*p[i]); adj[i]=min(1,run)
    return adj
def cramers_v(ct):
    chi=stats.chi2_contingency(ct,correction=False)[0]; return float(np.sqrt(chi/(ct.values.sum()*(min(ct.shape)-1))))
print("Merged datasets:",DATA_DIR.resolve())

Merged datasets: <repository>/week-02/reports/week_02/merged


## Part 0. Load and integrity checks

In [2]:
lines=pd.read_csv(DATA_DIR/"sales_orders_full.csv",low_memory=False,parse_dates=["order_date","delivery_date"])
check("sales lines",130402,len(lines),len(lines)==130402)
o=lines.drop_duplicates("so_id").copy()
check("unique orders",20000,len(o),len(o)==20000)
check("order status has only Delivered and Cancelled",["Cancelled","Delivered"],sorted(o.order_status.unique()),set(o.order_status)=={"Cancelled","Delivered"})
check("status is the same on every line of an order",0,int((lines.groupby("so_id").order_status.nunique()>1).sum()),(lines.groupby("so_id").order_status.nunique()>1).sum()==0)
o["can"]=(o.order_status=="Cancelled").astype(int)
nc=int(o.can.sum()); rate=nc/len(o)
check("cancelled orders match Week 3 (1,967)",1967,nc,nc==1967)
check("cancellation rate is 9.8% (Week 3)",9.8,round(rate*100,1),round(rate*100,1)==9.8)
val_c=o.loc[o.can==1,"total_order_value"].sum(); val_all=o.total_order_value.sum()
check("order value equals the sum of its lines",0,round(float(lines.groupby("so_id").line_total.sum().reindex(o.so_id).values.sum()-val_all),2),abs(lines.groupby("so_id").line_total.sum().reindex(o.so_id).values.sum()-val_all)<1)
print(f"{nc:,} of {len(o):,} orders cancelled ({rate*100:.2f}%), worth Rs {val_c/1e9:.2f}B of Rs {val_all/1e9:.2f}B placed ({val_c/val_all*100:.2f}% of value)")
o["year"]=o.order_date.dt.year; o["month"]=o.order_date.dt.month
o["lines"]=o.so_id.map(lines.groupby("so_id").size())

1,967 of 20,000 orders cancelled (9.84%), worth Rs 2.49B of Rs 25.45B placed (9.78% of value)


## Part 1. The time picture
The six yearly rates, tested two ways: *are all years equal* (chi-square) and *is there a steady upward trend* (Cochran-Armitage, which uses the order of the years and is the more suitable test for a trend).

In [3]:
yr=o.groupby("year").can.agg(["sum","count"]); yr["rate_pct"]=yr["sum"]/yr["count"]*100
ci=[wilson(x,n) for x,n in zip(yr["sum"],yr["count"])]; yr["ci_low_pct"]=[c[0]*100 for c in ci]; yr["ci_high_pct"]=[c[1]*100 for c in ci]
chi_y,p_y,_,_=stats.chi2_contingency(pd.crosstab(o.year,o.can),correction=False)
def ca_trend(x,n,t):
    x,n,t=map(np.asarray,(x,n,t)); N=n.sum(); p=x.sum()/N
    T=(t*(x-n*p)).sum(); V=p*(1-p)*((n*t*t).sum()-((n*t).sum())**2/N); z=T/np.sqrt(V); return z,2*stats.norm.sf(abs(z))
z_t,p_t=ca_trend(yr["sum"],yr["count"],np.arange(len(yr)))
first,last=yr.iloc[0],yr.iloc[-1]
tab=pd.crosstab(o.year.isin([2024]),o.can); p_e=stats.fisher_exact(tab.values)[1]
print(yr.round(2).to_string())
print(f"\nAre all years equal? chi-square p = {p_y:.3f} (Cramer's V {cramers_v(pd.crosstab(o.year,o.can)):.3f})")
print(f"Steady upward trend? Cochran-Armitage z = {z_t:.2f}, p = {p_t:.3f} (rate rises about {(yr.rate_pct.iloc[-1]-yr.rate_pct.iloc[0])/5:.2f} points a year)")
print(f"2024 ({last.rate_pct:.2f}%) against 2019-2023 pooled ({o[o.year<2024].can.mean()*100:.2f}%): Fisher p = {p_e:.3f}; 2024 was singled out after looking at the data, so treat this as supporting, not confirming")
yr.round(3).to_csv(OUT/"q3_cancellation_by_year.csv")
check("yearly rates are 9.1% (2019) and 10.9% (2024), as in Week 4",(9.1,10.9),(round(first.rate_pct,1),round(last.rate_pct,1)),(round(first.rate_pct,1),round(last.rate_pct,1))==(9.1,10.9))
check("years are not significantly different as a group (chi-square p > 0.05)",">0.05",round(p_y,3),p_y>0.05)
check("trend test result is reproducible (p between 0.02 and 0.04, Week 4 value 0.030)","0.02-0.04",round(p_t,3),0.02<p_t<0.04)

      sum  count  rate_pct  ci_low_pct  ci_high_pct
year                                               
2019  305   3336      9.14        8.21        10.17
2020  320   3358      9.53        8.58        10.57
2021  327   3365      9.72        8.76        10.76
2022  331   3279     10.09        9.11        11.17
2023  324   3346      9.68        8.73        10.73
2024  360   3316     10.86        9.84        11.96

Are all years equal? chi-square p = 0.265 (Cramer's V 0.018)
Steady upward trend? Cochran-Armitage z = 2.18, p = 0.030 (rate rises about 0.34 points a year)
2024 (10.86%) against 2019-2023 pooled (9.63%): Fisher p = 0.032; 2024 was singled out after looking at the data, so treat this as supporting, not confirming


In [4]:
mo=o.groupby("month").can.agg(["sum","count"]); mo["rate_pct"]=mo["sum"]/mo["count"]*100
cim=[wilson(x,n) for x,n in zip(mo["sum"],mo["count"])]; mo["ci_low_pct"]=[c[0]*100 for c in cim]; mo["ci_high_pct"]=[c[1]*100 for c in cim]
chi_m,p_m,_,_=stats.chi2_contingency(pd.crosstab(o.month,o.can),correction=False)
print(mo.round(2).to_string()); print(f"\nMonth of year, all 12 equal? chi-square p = {p_m:.3f}, V = {cramers_v(pd.crosstab(o.month,o.can)):.3f} (this is one of many cuts; see the Holm adjustment in Part 2)")
mo.round(3).to_csv(OUT/"q3_cancellation_by_month.csv")

       sum  count  rate_pct  ci_low_pct  ci_high_pct
month                                               
1      165   1696      9.73        8.41        11.23
2      155   1513     10.24        8.82        11.87
3      172   1681     10.23        8.87        11.77
4      170   1611     10.55        9.14        12.15
5      167   1718      9.72        8.41        11.21
6      181   1618     11.19        9.74        12.82
7      128   1723      7.43        6.28         8.76
8      169   1723      9.81        8.49        11.30
9      182   1686     10.79        9.40        12.37
10     158   1664      9.50        8.18        11.00
11     149   1699      8.77        7.52        10.21
12     171   1668     10.25        8.89        11.80

Month of year, all 12 equal? chi-square p = 0.038, V = 0.032 (this is one of many cuts; see the Holm adjustment in Part 2)


## Part 2. The segment scan
Every cut available on an order, one test each, adjusted together.

In [5]:
o["value_band"]=pd.qcut(o.total_order_value,4,labels=["1 low","2","3","4 high"]).astype(str)
o["lines_band"]=pd.cut(o.lines,[0,4,6,8,100],labels=["1-4 lines","5-6","7-8","9+"]).astype(str)
dims={"Customer region":"customer_region","Branch region":"branch_region","Customer type":"customer_type","Sales channel":"sales_channel",
      "Payment terms":"payment_terms","Industry segment":"industry_segment","Customer rating":"customer_rating","Warehouse type":"warehouse_type",
      "Order value quartile":"value_band","Lines per order":"lines_band","Order year":"year","Order month":"month"}
rows=[];seg=[]
for label,col in dims.items():
    ct=pd.crosstab(o[col],o.can); chi,p,_,_=stats.chi2_contingency(ct,correction=False)
    g=o.groupby(col).can.agg(["sum","count"]); g["rate"]=g["sum"]/g["count"]*100
    rows.append(dict(dimension=label,segments=len(g),smallest_segment_orders=int(g["count"].min()),lowest_rate_pct=g.rate.min(),highest_rate_pct=g.rate.max(),spread_points=g.rate.max()-g.rate.min(),chi2=chi,p_value=p,cramers_v=cramers_v(ct)))
    for k,r in g.iterrows():
        lo,hi=wilson(r["sum"],r["count"]); seg.append(dict(dimension=label,segment=str(k),orders=int(r["count"]),cancelled=int(r["sum"]),rate_pct=r["rate"],ci_low_pct=lo*100,ci_high_pct=hi*100))
T=pd.DataFrame(rows); T["p_holm_adjusted"]=holm(T.p_value); T=T.sort_values("p_value")
S=pd.DataFrame(seg)
T.round(4).to_csv(OUT/"q3_segment_tests.csv",index=False); S.round(3).to_csv(OUT/"q3_segment_rates.csv",index=False)
print(T.round(3).to_string(index=False))
print(f"\nSmallest p-value {T.p_value.min():.3f} ({T.iloc[0].dimension}); after adjusting for {len(T)} cuts it is {T.p_holm_adjusted.min():.2f}. Largest effect size: V = {T.cramers_v.max():.3f} (0.1 is conventionally 'small').")
check("number of segment cuts tested",12,len(T),len(T)==12)
check("no cut is significant after Holm adjustment (all adjusted p > 0.05)",">0.05",round(T.p_holm_adjusted.min(),3),T.p_holm_adjusted.min()>0.05)
check("every effect size is negligible (Cramer's V < 0.05)","<0.05",round(T.cramers_v.max(),3),T.cramers_v.max()<0.05)
check("segment orders add up to 20,000 in every cut",20000,sorted(S.groupby("dimension").orders.sum().unique()),(S.groupby("dimension").orders.sum()==20000).all())

           dimension  segments  smallest_segment_orders  lowest_rate_pct  highest_rate_pct  spread_points   chi2  p_value  cramers_v  p_holm_adjusted
         Order month        12                     1513            7.429            11.187          3.758 20.619    0.038      0.032             0.45
     Lines per order         4                     3301            9.454            10.845          1.391  4.999    0.172      0.016             1.00
          Order year         6                     3279            9.143            10.856          1.714  6.446    0.265      0.018             1.00
      Warehouse type         4                     2953            9.177            10.226          1.048  2.780    0.427      0.012             1.00
     Customer region         6                      422            9.121            10.370          1.249  4.834    0.436      0.016             1.00
     Customer rating         5                     3519            9.313            10.381          

## Part 2b. Is the scan as a whole beyond chance? (family-wise permutation test)
The Holm adjustment is conservative. A permutation test answers the same question directly: shuffle the cancelled/delivered labels across orders 5,000 times, run the whole 12-cut scan each time, and record the smallest p-value found. If real concentration existed, the observed smallest p-value would be rarer than that.

In [6]:
rng=np.random.default_rng(SEED)
dimcols=list(dims.values()); codes={c:pd.factorize(o[c])[0] for c in dimcols}; ncat={c:codes[c].max()+1 for c in dimcols}; yv=o.can.values; Ntot=len(o)
def scan_minp(y):
    ps=[]
    for c in dimcols:
        k=ncat[c]; cnt=np.bincount(codes[c],minlength=k); x=np.bincount(codes[c],weights=y,minlength=k); r=y.sum()/Ntot
        ex1=cnt*r; ex0=cnt*(1-r); chi=(((x-ex1)**2)/ex1+(((cnt-x)-ex0)**2)/ex0).sum(); ps.append(stats.chi2.sf(chi,k-1))
    return min(ps)
obs=scan_minp(yv); null=np.array([scan_minp(rng.permutation(yv)) for _ in range(NBOOT)])
p_fw=(1+(null<=obs).sum())/(NBOOT+1)
print(f"Smallest p-value in the real scan: {obs:.3f}. Across {NBOOT:,} shuffled scans, a smallest p-value that low or lower appears {p_fw*100:.0f}% of the time (family-wise p = {p_fw:.2f}). Typical shuffled minimum: {np.median(null):.3f}.")
pd.DataFrame([["smallest p-value, real scan",obs],["family-wise permutation p",p_fw],["median smallest p, shuffled scans",np.median(null)],["5th percentile smallest p, shuffled scans",np.percentile(null,5)]],columns=["measure","value"]).round(4).to_csv(OUT/"q3_scan_permutation.csv",index=False)
check("the real scan is no stronger than shuffled scans (family-wise permutation p > 0.05)",">0.05",round(p_fw,3),p_fw>0.05)
check("permutation and Holm agree on the verdict (both non-significant)",True,(round(p_fw,2),round(T.p_holm_adjusted.min(),2)),p_fw>0.05 and T.p_holm_adjusted.min()>0.05)

Smallest p-value in the real scan: 0.038. Across 5,000 shuffled scans, a smallest p-value that low or lower appears 37% of the time (family-wise p = 0.37). Typical shuffled minimum: 0.058.


## Part 2c. How large a gap can be ruled out?
For every segment with at least 1,000 orders, the upper end of its 95% interval minus the overall rate is the largest excess cancellation the data still allows.

In [7]:
big=S[(S.orders>=1000)&(S.dimension.isin(["Customer region","Customer type","Sales channel","Payment terms","Industry segment","Customer rating","Warehouse type","Branch region","Order value quartile","Lines per order"]))].copy()
big["max_excess_points"]=big.ci_high_pct-rate*100; big["max_shortfall_points"]=rate*100-big.ci_low_pct
worst=big.sort_values("max_excess_points",ascending=False).iloc[0]
print(f"{len(big)} segments with 1,000+ orders. The largest excess still allowed is {big.max_excess_points.max():.1f} points ({worst.dimension}: {worst.segment}); the typical allowance is {big.max_excess_points.median():.1f} points.")
print(f"So no segment of 1,000+ orders cancels more than about {rate*100+big.max_excess_points.max():.0f}% (against {rate*100:.1f}% overall) at 95% confidence.")
big.round(3).to_csv(OUT/"q3_ruled_out_gaps.csv",index=False)
rng=np.random.default_rng(SEED); a=o.loc[o.year==2024,"can"].values; b=o.loc[o.year==2019,"can"].values
diff=np.array([rng.choice(a,len(a)).mean()-rng.choice(b,len(b)).mean() for _ in range(NBOOT)])*100; dlo,dhi=np.percentile(diff,[2.5,97.5]); dpt=(a.mean()-b.mean())*100
print(f"\nRise from 2019 to 2024: {dpt:.2f} points, bootstrap 95% interval [{dlo:.2f}, {dhi:.2f}] points. A rise of up to about {dhi:.1f} points is consistent with the data; so is a rise close to zero.")
pd.DataFrame([["2019 to 2024 change (points)",dpt],["bootstrap 95% low",dlo],["bootstrap 95% high",dhi]],columns=["measure","value"]).round(3).to_csv(OUT/"q3_rise_interval.csv",index=False)
check("no segment of 1,000+ orders can be cancelling more than 3 points above the overall rate","<3",round(big.max_excess_points.max(),2),big.max_excess_points.max()<3)
check("the 2019-2024 rise interval is wide enough to include zero or very small values (lower bound under 0.5 points)","<0.5",round(dlo,2),dlo<0.5)

45 segments with 1,000+ orders. The largest excess still allowed is 2.1 points (Lines per order: 7-8); the typical allowance is 0.8 points.
So no segment of 1,000+ orders cancels more than about 12% (against 9.8% overall) at 95% confidence.



Rise from 2019 to 2024: 1.71 points, bootstrap 95% interval [0.25, 3.13] points. A rise of up to about 3.1 points is consistent with the data; so is a rise close to zero.


## Part 3. Products and categories
An order holds about 6.5 lines across different products, so a product cannot own an order. The test asks: *are orders that contain this product cancelled more often than orders that do not?* Thirty product tests are adjusted together.

In [8]:
def exposure(col):
    out=[]
    for k,gr in lines.groupby(col):
        ids=gr.so_id.unique(); inn=o.so_id.isin(ids); a=o.loc[inn,"can"]; b=o.loc[~inn,"can"]
        t=pd.crosstab(inn,o.can); chi,p,_,_=stats.chi2_contingency(t,correction=False)
        out.append(dict(item=k,orders_containing=int(inn.sum()),rate_with_pct=a.mean()*100,rate_without_pct=b.mean()*100,difference_points=(a.mean()-b.mean())*100,p_value=p))
    R=pd.DataFrame(out); R["p_holm_adjusted"]=holm(R.p_value); return R.sort_values("p_value")
P=exposure("product_id"); names=lines.drop_duplicates("product_id").set_index("product_id").product_name
P.insert(1,"product_name",P["item"].map(names)); K=exposure("category")
P.round(4).to_csv(OUT/"q3_product_exposure_tests.csv",index=False); K.round(4).to_csv(OUT/"q3_category_exposure_tests.csv",index=False)
print(P.head(5).round(3).to_string(index=False)); print(f"\n30 products: raw p-values below 0.05 = {(P.p_value<0.05).sum()} (about {0.05*len(P):.1f} expected by chance alone); smallest adjusted p = {P.p_holm_adjusted.min():.2f}")
print(f"{len(K)} categories: smallest raw p = {K.p_value.min():.3f}, adjusted {K.p_holm_adjusted.min():.2f}")
print(f"Product exposure rates span {P.rate_with_pct.min():.1f}% to {P.rate_with_pct.max():.1f}%")
check("30 products tested",30,len(P),len(P)==30)
check("no product is significant after Holm adjustment",">0.05",round(P.p_holm_adjusted.min(),3),P.p_holm_adjusted.min()>0.05)
check("no category is significant after Holm adjustment",">0.05",round(K.p_holm_adjusted.min(),3),K.p_holm_adjusted.min()>0.05)

item          product_name  orders_containing  rate_with_pct  rate_without_pct  difference_points  p_value  p_holm_adjusted
P003     Air Filter AF-120               3895          8.755            10.096             -1.341    0.012            0.349
P028  Light Assembly LA-18               3917         10.697             9.625              1.072    0.043            1.000
P030      Fuel Pump FP-350               3856          8.999            10.035             -1.036    0.052            1.000
P001 Hydraulic Pump HP-300               3854          9.211             9.984             -0.773    0.148            1.000
P026 Grease Cartridge GC-5               3823         10.332             9.718              0.615    0.251            1.000

30 products: raw p-values below 0.05 = 2 (about 1.5 expected by chance alone); smallest adjusted p = 0.35
15 categories: smallest raw p = 0.050, adjusted 0.76
Product exposure rates span 8.8% to 10.7%


## Part 4. Are some customers cancelling more than others?
If cancellation had a customer-level driver, customers would differ more than chance allows. The test compares the spread of the 500 customers' cancellation counts with what a constant 9.8% rate would produce.

In [9]:
cu=o.groupby("customer_id").can.agg(["sum","count"]); p0=cu["sum"].sum()/cu["count"].sum(); ex=cu["count"]*p0
chi_c=float(((cu["sum"]-ex)**2/(ex*(1-p0))).sum()); df=len(cu)-1; p_c=float(stats.chi2.sf(chi_c,df))
print(f"500 customers, {cu['count'].min()}-{cu['count'].max()} orders each; cancellation counts per customer vs a constant {p0*100:.2f}% rate: chi-square {chi_c:.0f} on {df} df, p = {p_c:.2f}")
print(f"Highest customer rate {(cu['sum']/cu['count']).max()*100:.1f}%, lowest {(cu['sum']/cu['count']).min()*100:.1f}% (with ~40 orders each, a spread this wide is expected by chance)")
pd.DataFrame([["customers",len(cu)],["cancellation rate used (%)",p0*100],["chi-square",chi_c],["degrees of freedom",df],["p-value",p_c],["highest customer rate (%)",(cu["sum"]/cu["count"]).max()*100],["lowest customer rate (%)",(cu["sum"]/cu["count"]).min()*100]],columns=["measure","value"]).round(4).to_csv(OUT/"q3_customer_heterogeneity.csv",index=False)
check("customers do not differ more than chance (p > 0.05)",">0.05",round(p_c,3),p_c>0.05)

500 customers, 26-63 orders each; cancellation counts per customer vs a constant 9.84% rate: chi-square 468 on 499 df, p = 0.84
Highest customer rate 25.0%, lowest 0.0% (with ~40 orders each, a spread this wide is expected by chance)


## Part 5. Do cancelled orders look different from delivered ones?

In [10]:
can=o[o.can==1]; dlv=o[o.can==0]
dd=lines.drop_duplicates("so_id").assign(days=lambda d:(d.delivery_date-d.order_date).dt.days)
rowsC=[]
for lab,col in [("Order value (Rs)","total_order_value"),("Lines per order","lines")]:
    u=stats.mannwhitneyu(can[col],dlv[col]); rowsC.append(dict(measure=lab,cancelled_mean=can[col].mean(),delivered_mean=dlv[col].mean(),cancelled_median=can[col].median(),delivered_median=dlv[col].median(),p_value=u.pvalue))
dc=dd[dd.so_id.isin(can.so_id)].days; dv=dd[dd.so_id.isin(dlv.so_id)].days; u=stats.mannwhitneyu(dc,dv)
rowsC.append(dict(measure="Days from order to recorded delivery date",cancelled_mean=dc.mean(),delivered_mean=dv.mean(),cancelled_median=dc.median(),delivered_median=dv.median(),p_value=u.pvalue))
CD=pd.DataFrame(rowsC); CD.round(4).to_csv(OUT/"q3_cancelled_vs_delivered.csv",index=False); print(CD.round(3).to_string(index=False))
has_dd=float(can.delivery_date.notna().mean())
print(f"\n{has_dd*100:.0f}% of cancelled orders still carry a delivery date, and it follows the same lead time as delivered orders. A cancelled order should have no delivery. This looks like a data-generation or system artefact and is a question for the data owner (see Part 7).")
check("cancelled orders are not larger or smaller than delivered ones (order value p > 0.05)",">0.05",round(CD.p_value.iloc[0],3),CD.p_value.iloc[0]>0.05)
check("all cancelled orders carry a delivery date (data quality flag)",1.0,round(has_dd,3),has_dd==1.0)

                                  measure  cancelled_mean  delivered_mean  cancelled_median  delivered_median  p_value
                         Order value (Rs)     1265076.324     1273165.166         1009600.0         1014840.0    0.528
                          Lines per order           6.480           6.524               7.0               7.0    0.585
Days from order to recorded delivery date           8.052           8.063               8.0               8.0    0.936

100% of cancelled orders still carry a delivery date, and it follows the same lead time as delivered orders. A cancelled order should have no delivery. This looks like a data-generation or system artefact and is a question for the data owner (see Part 7).


## Part 6. What size of difference could this analysis have detected?
A non-significant result is only informative if the test could have seen a real gap. This is the gap a segment would need to show, against the rest, to be detected 80% of the time at the 5% level (no multiple-testing adjustment, so it is the optimistic case).

In [11]:
z=stats.norm.ppf(0.975)+stats.norm.ppf(0.80)
def mde(n1,n2,p=rate): return z*np.sqrt(p*(1-p)*(1/n1+1/n2))*100
sz=S[S.dimension.isin(["Customer region","Customer type","Sales channel","Payment terms"])]
mins=sz.orders.min(); med=sz.orders.median()
print(f"Smallest of these segments: {mins:,} orders -> detectable gap about {mde(mins,len(o)-mins):.1f} points; median segment ({med:,.0f} orders) -> about {mde(med,len(o)-med):.1f} points")
sm=sz.loc[sz.orders.idxmin()]
print(f"Smallest segment is {sm.dimension} = {sm.segment}. A gap of about {mde(med,len(o)-med):.1f} points would have been seen in a typical segment; in the smallest segment only gaps of about {mde(mins,len(o)-mins):.1f} points or more would. Smaller gaps cannot be ruled out, especially for the smallest regions.")
pd.DataFrame([["smallest region/type/channel/terms segment (orders)",mins,mde(mins,len(o)-mins)],["median segment (orders)",med,mde(med,len(o)-med)]],columns=["segment","orders","detectable_gap_points_80pct_power"]).round(2).to_csv(OUT/"q3_detectable_gap.csv",index=False)
check("a typical segment could detect a gap under 3 points (the smallest only about 4)", "<3 (median segment)",round(mde(med,len(o)-med),2),mde(med,len(o)-med)<3)
check("even the smallest segment could detect a gap under 5 points","<5",round(mde(mins,len(o)-mins),2),mde(mins,len(o)-mins)<5)

Smallest of these segments: 422 orders -> detectable gap about 4.1 points; median segment (4,124 orders) -> about 1.5 points
Smallest segment is Customer region = North East. A gap of about 1.5 points would have been seen in a typical segment; in the smallest segment only gaps of about 4.1 points or more would. Smaller gaps cannot be ruled out, especially for the smallest regions.


## Charts

In [12]:
cov=S[S.dimension.isin(["Customer region","Customer type","Sales channel","Payment terms","Order value quartile","Customer rating"])]
check("every segment shown in the segment chart has an interval that covers the overall rate",True,bool(((cov.ci_low_pct<=rate*100)&(cov.ci_high_pct>=rate*100)).all()),bool(((cov.ci_low_pct<=rate*100)&(cov.ci_high_pct>=rate*100)).all()))
fig,ax=plt.subplots(figsize=(8.2,4.2),facecolor=SURF); ax.set_facecolor(SURF)
ax.errorbar(yr.index,yr.rate_pct,yerr=[yr.rate_pct-yr.ci_low_pct,yr.ci_high_pct-yr.rate_pct],fmt="o-",color=BLUE,ecolor=MUTED,capsize=4,lw=2,ms=7)
ax.axhline(rate*100,color=ORANGE,ls="--",lw=1.4)
ax.text(2019.05,rate*100+0.12,f"All years: {rate*100:.2f}%",color=ORANGE,fontsize=9)
for x,y in zip(yr.index,yr.rate_pct): ax.text(x,y+0.75,f"{y:.1f}%",ha="center",fontsize=9,color=INK2)
ax.set_ylim(6,14.5); ax.set_ylabel("Cancelled orders (%)",color=INK2); ax.grid(axis="y",color=GRID,lw=.6); ax.set_axisbelow(True)
[s.set_visible(False) for k,s in ax.spines.items() if k in("top","right")]
ax.set_title("Cancellation by year: a small rise inside wide uncertainty",loc="left",color=INK,fontsize=12,fontweight="bold")
ax.text(0.0,-0.18,f"Bars show 95% intervals. Equal-years test p = {p_y:.2f}; trend test p = {p_t:.2f}.",transform=ax.transAxes,fontsize=8.5,color=INK2)
plt.tight_layout(); plt.savefig(CH/"q3_01_cancellation_by_year.png",dpi=150,facecolor=SURF); plt.close()

In [13]:
show=["Customer region","Customer type","Sales channel","Payment terms","Order value quartile","Customer rating"]
fig,axes=plt.subplots(2,3,figsize=(11,6.2),facecolor=SURF,sharex=True)
for ax,dname in zip(axes.ravel(),show):
    d=S[S.dimension==dname].reset_index(drop=True); y=np.arange(len(d))[::-1]
    ax.set_facecolor(SURF); ax.axvline(rate*100,color=ORANGE,ls="--",lw=1.2)
    ax.errorbar(d.rate_pct,y,xerr=[d.rate_pct-d.ci_low_pct,d.ci_high_pct-d.rate_pct],fmt="o",color=BLUE,ecolor=MUTED,capsize=3,ms=5)
    ax.set_yticks(y); ax.set_yticklabels(d.segment,fontsize=8.5,color=INK2); ax.set_title(dname,loc="left",fontsize=10,color=INK,fontweight="bold")
    ax.grid(axis="x",color=GRID,lw=.6); ax.set_axisbelow(True); [s.set_visible(False) for k,s in ax.spines.items() if k in("top","right")]
    ax.set_xlim(6.5,13.5)
for ax in axes[1]: ax.set_xlabel("Cancelled orders (%)",color=INK2,fontsize=9)
fig.suptitle("Every segment's interval covers the overall rate (dashed line)",x=0.01,ha="left",fontsize=12,fontweight="bold",color=INK)
plt.tight_layout(rect=[0,0,1,0.95]); plt.savefig(CH/"q3_02_segment_rates.png",dpi=150,facecolor=SURF); plt.close()

In [14]:
fig,ax=plt.subplots(figsize=(8.2,4.0),facecolor=SURF); ax.set_facecolor(SURF)
ax.errorbar(mo.index,mo.rate_pct,yerr=[mo.rate_pct-mo.ci_low_pct,mo.ci_high_pct-mo.rate_pct],fmt="o-",color=BLUE,ecolor=MUTED,capsize=3,lw=1.6,ms=5)
ax.axhline(rate*100,color=ORANGE,ls="--",lw=1.2); ax.set_xticks(range(1,13)); ax.set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"],color=INK2)
ax.set_ylim(5,15); ax.set_ylabel("Cancelled orders (%)",color=INK2); ax.grid(axis="y",color=GRID,lw=.6); ax.set_axisbelow(True)
[s.set_visible(False) for k,s in ax.spines.items() if k in("top","right")]
ax.set_title("Cancellation by month of year: no pattern that survives adjustment",loc="left",color=INK,fontsize=12,fontweight="bold")
ax.text(0.0,-0.16,f"Raw 12-month test p = {p_m:.3f}; adjusted for the 12 cuts examined it is {T.loc[T.dimension=='Order month','p_holm_adjusted'].iloc[0]:.2f}.",transform=ax.transAxes,fontsize=8.5,color=INK2)
plt.tight_layout(); plt.savefig(CH/"q3_03_cancellation_by_month.png",dpi=150,facecolor=SURF); plt.close()

## Q3 answer

In [15]:
topdim=T.iloc[0]
print("Q3 ANSWER")
print(f"- Cancellation is NOT concentrated in any region, customer type, channel, payment term, industry, rating, warehouse, order size or product.")
print(f"- Across 12 cuts the largest effect size is V = {T.cramers_v.max():.3f}; the smallest raw p-value ({topdim.dimension}, p = {topdim.p_value:.3f}) is not significant after adjustment (p = {topdim.p_holm_adjusted:.2f}).")
print(f"- Across 30 products, {(P.p_value<0.05).sum()} raw p-values fall below 0.05, which is what chance alone produces; none survives adjustment.")
print(f"- Customers do not differ more than chance (p = {p_c:.2f}); cancelled orders are the same size and shape as delivered ones.")
print(f"- The only time signal is a weak upward trend ({first.rate_pct:.1f}% in 2019 to {last.rate_pct:.1f}% in 2024; trend p = {p_t:.3f}, equal-years p = {p_y:.2f}).")
print(f"- Scan as a whole: shuffling labels gives an equally strong smallest p-value {p_fw*100:.0f}% of the time (family-wise p = {p_fw:.2f}). For segments of 1,000+ orders, none can be cancelling more than about {big.max_excess_points.max():.1f} points above the overall rate.")
print("- Caveat: no cancellation-reason field exists, and cancelled orders carry a delivery date. Cause cannot be established from this data.")

Q3 ANSWER
- Cancellation is NOT concentrated in any region, customer type, channel, payment term, industry, rating, warehouse, order size or product.
- Across 12 cuts the largest effect size is V = 0.032; the smallest raw p-value (Order month, p = 0.038) is not significant after adjustment (p = 0.45).
- Across 30 products, 2 raw p-values fall below 0.05, which is what chance alone produces; none survives adjustment.
- Customers do not differ more than chance (p = 0.84); cancelled orders are the same size and shape as delivered ones.
- The only time signal is a weak upward trend (9.1% in 2019 to 10.9% in 2024; trend p = 0.030, equal-years p = 0.27).
- Scan as a whole: shuffling labels gives an equally strong smallest p-value 37% of the time (family-wise p = 0.37). For segments of 1,000+ orders, none can be cancelling more than about 2.1 points above the overall rate.
- Caveat: no cancellation-reason field exists, and cancelled orders carry a delivery date. Cause cannot be established fr

## Part 7. Questions for the data owner
1. What does the `Cancelled` status mean in the source system (customer cancelled, rejected at credit check, stock-out, system status)?
2. Why do all cancelled orders carry a delivery date with the usual lead time?
3. Is a cancellation reason or cancellation date recorded anywhere?

## Part 8. Validation summary

In [16]:
V=pd.DataFrame(checks); V.to_csv("validation_report_week05_q3.csv",index=False)
print(f"{(V.Status=='PASS').sum()} of {len(V)} checks pass"); V

27 of 27 checks pass


,Check_Item,Expected,Actual,Status
0,sales lines,130402,130402,PASS
1,unique orders,20000,20000,PASS
2,order status has only Delivered and Cancelled,"[Cancelled, Delivered]","[Cancelled, Delivered]",PASS
3,status is the same on every line of an order,0,0,PASS
4,"cancelled orders match Week 3 (1,967)",1967,1967,PASS
5,cancellation rate is 9.8% (Week 3),9.8,9.8,PASS
6,order value equals the sum of its lines,0,0.0,PASS
7,"yearly rates are 9.1% (2019) and 10.9% (2024),...","(9.1, 10.9)","(9.1, 10.9)",PASS
8,years are not significantly different as a gro...,>0.05,0.265,PASS
9,trend test result is reproducible (p between 0...,0.02-0.04,0.03,PASS
